# Embeddings Pipeline

Genera los embeddings semánticos y el índice FAISS para búsqueda por similaridad.

**Input:** `df_clean_final.parquet`
**Outputs:** `df_final_embeddings.parquet`, `recipe_embeddings.npy`, `recipe_faiss.index`

In [1]:
# ==========================================
# 📦 INSTALLS
# ==========================================
!pip install sentence-transformers faiss-cpu pyarrow numpy tqdm -q
print("Installs OK")

Installs OK


In [2]:
# ==========================================
# 📦 IMPORTS
# ==========================================
import json
import ast
import re
import os
import numpy as np
import pandas as pd
import faiss
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from IPython.display import display

print("Imports OK")

Imports OK


---
## Carga del dataset

In [3]:
# ==========================================
# CARGAR df_clean_final.parquet
# ==========================================
parquet_path = "df_clean_final.parquet"
df = pd.read_parquet(parquet_path)

print(f"Shape: {df.shape}")
print(f"\nColumnas ({len(df.columns)}):")
for col in df.columns:
    n_null = df[col].isna().sum()
    print(f"  {col:<45}  null: {n_null:>6,}")

Shape: (62126, 41)

Columnas (41):
  recipe_title                                   null:      0
  category                                       null:      0
  subcategory                                    null:      0
  description                                    null:      0
  ingredients                                    null:      0
  directions                                     null:      0
  num_ingredients                                null:      0
  num_steps                                      null:      0
  ingredient_text                                null:      0
  directions_text                                null:      0
  combined_text                                  null:      0
  ingredients_raw                                null:      0
  directions_raw                                 null:      0
  ingredients_canonical                          null:      0
  cuisine_list                                   null:      0
  course_list                      

In [4]:
# ==========================================
# DETECCIÓN DE NOMBRES DE COLUMNAS
# Los nombres cambian según si el parquet viene del notebook
# de limpieza (DP_dataset_versionfinal) o del de imágenes.
# ==========================================

# Columna de ingredientes procesados
INGR_COL = next(
    (c for c in ["ingredients_text_processed", "ingredient_text"] if c in df.columns),
    None,
)
# Columna de perfil dietético
DIET_COL = next(
    (c for c in ["dietary_profile_updated", "dietary_profile"] if c in df.columns),
    None,
)

print(f"Columna de ingredientes : {INGR_COL}")
print(f"Columna perfil dietético: {DIET_COL}")

assert INGR_COL is not None, "No se encontró columna de ingredientes. Revisa el parquet."
assert DIET_COL is not None, "No se encontró columna de perfil dietético. Revisa el parquet."

print(f"\nMuestra de {INGR_COL}: {df[INGR_COL].iloc[0][:100]}")
print(f"Muestra de {DIET_COL}: {df[DIET_COL].iloc[0]}")
print(f"Muestra de cook_speed: {df['cook_speed'].value_counts().to_dict()}")

Columna de ingredientes : ingredient_text
Columna perfil dietético: dietary_profile

Muestra de ingredient_text: ketchup beer worcestershire sauce onion powder cayenne baking potatoes olive oil cooking spray garli
Muestra de dietary_profile: ["vegan", "gluten_free", "nut_free", "kosher"]
Muestra de cook_speed: {'fast': 47323, 'medium': 13556, 'slow': 1247}


---
## Paso 1 — Limpieza de columnas críticas

Filtros: `recipe_title` ≤ 200 chars · `num_ingredients ≥ 3` · `directions_text` ≥ 20 palabras · `dietary_profile` parseable como lista · `cook_speed` ∈ {fast, medium, slow, unspecified}.

In [5]:
# ==========================================
# PASO 1: LIMPIAR COLUMNAS CRÍTICAS
# ==========================================

def try_parse_json_list(s):
    """Intenta parsear una cadena como lista JSON/Python. Devuelve None si falla."""
    if pd.isna(s) or str(s).strip() in ("", "[]", "nan"):
        return None
    try:
        result = json.loads(s)
        return result if isinstance(result, list) else None
    except (json.JSONDecodeError, TypeError):
        try:
            result = ast.literal_eval(str(s))
            return result if isinstance(result, list) else None
        except Exception:
            return None


VALID_COOK_SPEEDS = {"fast", "medium", "slow", "unspecified"}
n_start = len(df)
print(f"Inicio: {n_start:,} registros\n")

# ── Filtro 1: recipe_title ────────────────────────────────────────────────────
mask_title = df["recipe_title"].notna() & (df["recipe_title"].str.len() <= 200)
n_removed = (~mask_title).sum()
df = df[mask_title].copy()
print(f"[1] recipe_title (no nulos, ≤200 chars): eliminados {n_removed:,} → {len(df):,} restantes")

# ── Filtro 2: ingredientes ≥ 3 ───────────────────────────────────────────────
n_prev = len(df)
mask_ingr = df[INGR_COL].notna()
if "num_ingredients" in df.columns:
    mask_ingr &= df["num_ingredients"] >= 3
else:
    # Fallback: contar palabras en el texto de ingredientes
    mask_ingr &= df[INGR_COL].str.split().str.len() >= 3
n_removed = (~mask_ingr).sum()
df = df[mask_ingr].copy()
print(f"[2] {INGR_COL} (no nulos, ≥3 ingredientes): eliminados {n_removed:,} → {len(df):,} restantes")

# ── Filtro 3: directions_text ≥ 20 palabras ──────────────────────────────────
n_prev = len(df)
mask_dir = df["directions_text"].notna() & (df["directions_text"].str.split().str.len() >= 20)
n_removed = (~mask_dir).sum()
df = df[mask_dir].copy()
print(f"[3] directions_text (no nulos, ≥20 palabras): eliminados {n_removed:,} → {len(df):,} restantes")

# ── Filtro 4: dietary_profile parseable como JSON list ───────────────────────
n_prev = len(df)
df["_diet_test"] = df[DIET_COL].apply(try_parse_json_list)
mask_diet = df["_diet_test"].notna()
n_removed = (~mask_diet).sum()
df = df[mask_diet].copy()
df = df.drop(columns=["_diet_test"])
print(f"[4] {DIET_COL} (parseable como JSON list): eliminados {n_removed:,} → {len(df):,} restantes")

# ── Filtro 5: cook_speed válido ───────────────────────────────────────────────
n_prev = len(df)
mask_speed = df["cook_speed"].isin(VALID_COOK_SPEEDS)
n_removed = (~mask_speed).sum()
df = df[mask_speed].copy()
print(f"[5] cook_speed (fast/medium/slow/unspecified): eliminados {n_removed:,} → {len(df):,} restantes")

print(f"\n{'='*50}")
print(f"Total eliminados: {n_start - len(df):,} ({(n_start - len(df))/n_start:.2%} del total)")
print(f"Dataset limpio  : {len(df):,} registros")

df = df.reset_index(drop=True)

Inicio: 62,126 registros

[1] recipe_title (no nulos, ≤200 chars): eliminados 0 → 62,126 restantes
[2] ingredient_text (no nulos, ≥3 ingredientes): eliminados 1,090 → 61,036 restantes
[3] directions_text (no nulos, ≥20 palabras): eliminados 422 → 60,614 restantes
[4] dietary_profile (parseable como JSON list): eliminados 0 → 60,614 restantes
[5] cook_speed (fast/medium/slow/unspecified): eliminados 0 → 60,614 restantes

Total eliminados: 1,512 (2.43% del total)
Dataset limpio  : 60,614 registros


---
## Paso 2 — `embedding_text`

Template (sin `difficulty` — filtro post-retrieval):

```
Title: {recipe_title}
Category: {category} | Subcategory: {subcategory}
Description: {description}
Ingredients: {ingredient_text}
Cuisine: {cuisine_list_text}
Course: {course_list_text}
Tastes: {tastes_text}
Dietary: {dietary_profile_updated_text}
Health flags: {health_flags_text}
Main ingredient: {main_ingredient}
Cook speed: {cook_speed}
Directions: {directions_text}
```

In [6]:
# ==========================================
# PASO 2A: COLUMNAS DE TEXTO DERIVADAS
# Convierte listas JSON → texto legible para el template
# ==========================================

def list_field_to_text(val):
    """Convierte una lista JSON/Python a string con comas. 'none' si vacía."""
    if pd.isna(val) or str(val).strip() in ("", "[]", "nan"):
        return "none"
    try:
        items = json.loads(val)
    except (json.JSONDecodeError, TypeError):
        try:
            items = ast.literal_eval(str(val))
        except Exception:
            return str(val).strip()
    if not isinstance(items, list) or len(items) == 0:
        return "none"
    return ", ".join(str(x).replace("_", " ").strip() for x in items if x)


df["cuisine_list_text"]            = df["cuisine_list"].apply(list_field_to_text)
df["course_list_text"]             = df["course_list"].apply(list_field_to_text)
df["tastes_text"]                  = df["tastes"].apply(list_field_to_text)
df["dietary_profile_updated_text"] = df[DIET_COL].apply(list_field_to_text)
df["health_flags_text"]            = df["health_flags"].apply(list_field_to_text)

print("Columnas de texto derivadas — muestra fila 0:")
for col in [
    "cuisine_list_text", "course_list_text", "tastes_text",
    "dietary_profile_updated_text", "health_flags_text",
]:
    print(f"  {col:<35}: {df[col].iloc[0]}")

Columnas de texto derivadas — muestra fila 0:
  cuisine_list_text                  : american, american region, asian, european, greek, korean, mediterranean, middle eastern region
  course_list_text                   : sauce
  tastes_text                        : spicy, bitter, savory
  dietary_profile_updated_text       : vegan, gluten free, nut free, kosher
  health_flags_text                  : plant based, healthy fats, fried


In [7]:
# ==========================================
# PASO 2B: CONSTRUIR embedding_text
# ==========================================

def build_embedding_text(row):
    raw = (
        f"Title: {row['recipe_title']}\n"
        f"Category: {row['category']} | Subcategory: {row['subcategory']}\n"
        f"Description: {row['description']}\n"
        f"Ingredients: {row[INGR_COL]}\n"
        f"Cuisine: {row['cuisine_list_text']}\n"
        f"Course: {row['course_list_text']}\n"
        f"Tastes: {row['tastes_text']}\n"
        f"Dietary: {row['dietary_profile_updated_text']}\n"
        f"Health flags: {row['health_flags_text']}\n"
        f"Main ingredient: {row['main_ingredient']}\n"
        f"Cook speed: {row['cook_speed']}\n"
        f"Directions: {row['directions_text']}"
    )
    # Colapsar whitespace
    return " ".join(raw.split())


tqdm.pandas(desc="Construyendo embedding_text")
df["embedding_text"] = df.progress_apply(build_embedding_text, axis=1)

print(f"\nembedding_text construido para {len(df):,} recetas.")
print(f"\nMuestra (primeros 400 chars):")
print(df["embedding_text"].iloc[0][:400] + "...")

Construyendo embedding_text:   0%|          | 0/60614 [00:00<?, ?it/s]


embedding_text construido para 60,614 recetas.

Muestra (primeros 400 chars):
Title: Air Fryer Potato Slices with Dipping Sauce Category: Air Fryer Recipes | Subcategory: Air Fryer Recipes Description: These air fryer potato slices, served with a beer ketchup dipping sauce, are a tasty finger food somewhere between a French fry and a potato chip. Do take the time to make the dipping sauce—it's worth it. Ingredients: ketchup beer worcestershire sauce onion powder cayenne bak...


---
## Paso 3 — Longitud y truncado

Threshold: 512 word-tokens (approx. con `len(text.split())`). Para textos > 512: truncar `directions_text` al 60%.

In [8]:
# ==========================================
# PASO 3: ESTADÍSTICAS DE LONGITUD
# ==========================================
df["_n_tokens"] = df["embedding_text"].str.split().str.len()

mean_len   = df["_n_tokens"].mean()
p95_len    = df["_n_tokens"].quantile(0.95)
n_over_512 = int((df["_n_tokens"] > 512).sum())

print("Estadísticas de longitud de embedding_text:")
print(f"  Longitud media (tokens) : {mean_len:.1f}")
print(f"  Percentil 95  (tokens) : {p95_len:.0f}")
print(f"  Registros > 512 tokens  : {n_over_512:,} ({n_over_512/len(df):.2%})")

# Distribución por rangos
bins   = [0, 128, 256, 384, 512, 768, float("inf")]
labels = ["≤128", "129–256", "257–384", "385–512", "513–768", ">768"]
hist   = pd.cut(df["_n_tokens"], bins=bins, labels=labels, right=True)
print("\nDistribución por rango de tokens:")
for rng, cnt in hist.value_counts().sort_index().items():
    pct = cnt / len(df)
    bar = "█" * int(pct * 50)
    print(f"  {str(rng):<10} {cnt:>7,} ({pct:5.1%})  {bar}")

Estadísticas de longitud de embedding_text:
  Longitud media (tokens) : 234.1
  Percentil 95  (tokens) : 396
  Registros > 512 tokens  : 632 (1.04%)

Distribución por rango de tokens:
  ≤128         3,267 ( 5.4%)  ██
  129–256     38,314 (63.2%)  ███████████████████████████████
  257–384     15,483 (25.5%)  ████████████
  385–512      2,918 ( 4.8%)  ██
  513–768        623 ( 1.0%)  
  >768             9 ( 0.0%)  


In [9]:
# ==========================================
# PASO 3B: TRUNCAR directions_text PARA > 512 TOKENS
# ==========================================
if n_over_512 == 0:
    print("No hay registros > 512 tokens. No se necesita truncado.")
else:
    mask_over = df["_n_tokens"] > 512
    print(f"Truncando directions_text al 60% en {n_over_512:,} registros...")

    def truncate_directions_60pct(directions):
        words  = str(directions).split()
        cutoff = max(20, int(len(words) * 0.6))
        return " ".join(words[:cutoff])

    df.loc[mask_over, "directions_text"] = (
        df.loc[mask_over, "directions_text"].apply(truncate_directions_60pct)
    )

    # Reconstruir embedding_text solo para los truncados
    tqdm.pandas(desc="Reconstruyendo embedding_text (truncados)")
    df.loc[mask_over, "embedding_text"] = (
        df[mask_over].progress_apply(build_embedding_text, axis=1)
    )

    # Actualizar conteo de tokens
    df.loc[mask_over, "_n_tokens"] = (
        df.loc[mask_over, "embedding_text"].str.split().str.len()
    )

    n_still_over = int((df["_n_tokens"] > 512).sum())
    new_mean = df["_n_tokens"].mean()
    print(f"\nDespués del truncado:")
    print(f"  Registros aún > 512 tokens : {n_still_over:,}")
    print(f"  Nueva longitud media        : {new_mean:.1f} tokens")

df = df.drop(columns=["_n_tokens"])
print("\nColumna temporal _n_tokens eliminada.")

Truncando directions_text al 60% en 632 registros...


Reconstruyendo embedding_text (truncados):   0%|          | 0/632 [00:00<?, ?it/s]


Después del truncado:
  Registros aún > 512 tokens : 13
  Nueva longitud media        : 232.3 tokens

Columna temporal _n_tokens eliminada.


---
## Paso 4 — Embeddings y FAISS

Modelo: `sentence-transformers/all-MiniLM-L6-v2` (384d, L2-normalizado).
`IndexFlatIP`: inner product sobre vectores normalizados = cosine similarity exacta.

In [10]:
# ==========================================
# PASO 4A: CARGAR MODELO Y GENERAR EMBEDDINGS
# ==========================================
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Cargando modelo: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print(f"Modelo cargado. Dimensión de salida: {model.get_sentence_embedding_dimension()}")

texts = df["embedding_text"].tolist()
print(f"\nCodificando {len(texts):,} textos (batch_size=64)...")

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # L2-normaliza → inner product = cosine similarity
)

print(f"\nShape de embeddings : {embeddings.shape}")
print(f"Tipo de datos       : {embeddings.dtype}")
print(f"Norma vector[0]     : {np.linalg.norm(embeddings[0]):.6f}  (≈ 1.0 si normalizado)")
print(f"Norma vector[100]   : {np.linalg.norm(embeddings[100]):.6f}")

Cargando modelo: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado. Dimensión de salida: 384

Codificando 60,614 textos (batch_size=64)...


/tmp/ipykernel_31937/1495739620.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Modelo cargado. Dimensión de salida: {model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/948 [00:00<?, ?it/s]


Shape de embeddings : (60614, 384)
Tipo de datos       : float32
Norma vector[0]     : 1.000000  (≈ 1.0 si normalizado)
Norma vector[100]   : 1.000000


In [11]:
# ==========================================
# PASO 4B: GUARDAR EMBEDDINGS NUMPY
# ==========================================
emb_path = "recipe_embeddings.npy"
np.save(emb_path, embeddings)
size_mb = os.path.getsize(emb_path) / 1_048_576
print(f"Guardado: {emb_path}")
print(f"  Shape  : {embeddings.shape}")
print(f"  Tamaño : {size_mb:.1f} MB")
print(f"  Dtype  : {embeddings.dtype}")

Guardado: recipe_embeddings.npy
  Shape  : (60614, 384)
  Tamaño : 88.8 MB
  Dtype  : float32


In [12]:
# ==========================================
# PASO 4C: CONSTRUIR Y GUARDAR ÍNDICE FAISS
#
# IndexFlatIP: búsqueda exacta por inner product.
# Con vectores normalizados (norma = 1), el inner product
# es equivalente a la cosine similarity.
# ==========================================
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))

print(f"Índice FAISS construido:")
print(f"  Tipo      : IndexFlatIP (inner product)")
print(f"  Dimensión : {dim}")
print(f"  Vectores  : {index.ntotal:,}")

faiss_path = "recipe_faiss.index"
faiss.write_index(index, faiss_path)
size_mb = os.path.getsize(faiss_path) / 1_048_576
print(f"\nGuardado: {faiss_path}")
print(f"  Tamaño : {size_mb:.1f} MB")

Índice FAISS construido:
  Tipo      : IndexFlatIP (inner product)
  Dimensión : 384
  Vectores  : 60,614

Guardado: recipe_faiss.index
  Tamaño : 88.8 MB


---
## Paso 5 — Validación

Top 5 por query: `recipe_title`, `cook_speed`, `dietary_profile`, `similarity_score`.

In [13]:
# ==========================================
# PASO 5: VALIDACIÓN CON QUERIES DE PRUEBA
# ==========================================
queries = [
    "quiero algo rápido vegetariano con arroz",
    "need a gluten free dessert with chocolate",
    "easy chicken soup for cold weather",
]

K = 5
SEP = "=" * 65

for q_idx, query in enumerate(queries, 1):
    print(SEP)
    print(f"Query {q_idx}: \"{query}\"")
    print(SEP)

    q_emb = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    scores, indices = index.search(q_emb, K)

    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        row = df.iloc[idx]
        diet_text = list_field_to_text(row[DIET_COL])
        print(f"  #{rank}  similarity={score:.4f}")
        print(f"       recipe_title            : {row['recipe_title']}")
        print(f"       cook_speed              : {row['cook_speed']}")
        print(f"       dietary_profile_updated : {diet_text}")
        print()

print(SEP)

Query 1: "quiero algo rápido vegetariano con arroz"
  #1  similarity=0.5086
       recipe_title            : Savory Vegetarian Quinoa
       cook_speed              : medium
       dietary_profile_updated : vegan, gluten free, nut free, halal, kosher

  #2  similarity=0.4996
       recipe_title            : Easy Guacamole
       cook_speed              : fast
       dietary_profile_updated : vegan, gluten free, nut free, halal, kosher

  #3  similarity=0.4942
       recipe_title            : Easy Guacamole
       cook_speed              : fast
       dietary_profile_updated : vegan, gluten free, nut free, halal, kosher

  #4  similarity=0.4928
       recipe_title            : Easy Guacamole
       cook_speed              : fast
       dietary_profile_updated : vegan, gluten free, nut free, halal, kosher

  #5  similarity=0.4890
       recipe_title            : Instant Pot Mexican Quinoa
       cook_speed              : fast
       dietary_profile_updated : vegan, gluten free, nut free,

---
## Paso 6 — Guardar `df_final_embeddings.parquet`

In [14]:
# ==========================================
# PASO 6: GUARDAR df_final_embeddings.parquet
# ==========================================

# Eliminar columnas temporales del pipeline de imágenes que ya no se necesitan
drop_temp = ["title_norm"]  # columna auxiliar del join con Epicurious
df_out = df.drop(columns=[c for c in drop_temp if c in df.columns], errors="ignore")

out_path = "df_final_embeddings.parquet"
df_out.to_parquet(out_path, index=False, engine="pyarrow")
size_mb = os.path.getsize(out_path) / 1_048_576

print(f"Guardado: {out_path}")
print(f"  Shape  : {df_out.shape}")
print(f"  Tamaño : {size_mb:.1f} MB")

NEW_COLS = {
    "embedding_text", "cuisine_list_text", "course_list_text",
    "tastes_text", "dietary_profile_updated_text", "health_flags_text",
    "dish_image_path", "ingredient_images",
}

print(f"\nColumnas del DataFrame final ({len(df_out.columns)}):")
for col in df_out.columns:
    n_null  = int(df_out[col].isna().sum())
    pct_null = n_null / len(df_out)
    mark = " ◀ embedding pipeline" if col in NEW_COLS else ""
    print(f"  {col:<45}  null: {n_null:>6,} ({pct_null:5.1%}){mark}")

Guardado: df_final_embeddings.parquet
  Shape  : (60614, 46)
  Tamaño : 146.5 MB

Columnas del DataFrame final (46):
  recipe_title                                   null:      0 ( 0.0%)
  category                                       null:      0 ( 0.0%)
  subcategory                                    null:      0 ( 0.0%)
  description                                    null:      0 ( 0.0%)
  ingredients                                    null:      0 ( 0.0%)
  directions                                     null:      0 ( 0.0%)
  num_ingredients                                null:      0 ( 0.0%)
  num_steps                                      null:      0 ( 0.0%)
  ingredient_text                                null:      0 ( 0.0%)
  directions_text                                null:      0 ( 0.0%)
  combined_text                                  null:      0 ( 0.0%)
  ingredients_raw                                null:      0 ( 0.0%)
  directions_raw                           

In [15]:
# ==========================================
# RESUMEN FINAL
# ==========================================
print("=" * 65)
print("RESUMEN DEL PIPELINE DE EMBEDDINGS")
print("=" * 65)

files = [
    ("df_final_embeddings.parquet", "DataFrame limpio con embedding_text e imágenes"),
    ("recipe_embeddings.npy",       "Embeddings numpy (n × 384, float32, normalizado)"),
    ("recipe_faiss.index",          "Índice FAISS IndexFlatIP para búsqueda exacta"),
]

for fname, desc in files:
    if os.path.exists(fname):
        size_mb = os.path.getsize(fname) / 1_048_576
        print(f"  ✓ {fname:<35}  {size_mb:>7.1f} MB  — {desc}")
    else:
        print(f"  ✗ {fname:<35}  NO ENCONTRADO")

print()
emb = np.load("recipe_embeddings.npy")
print(f"  Recetas procesadas : {emb.shape[0]:,}")
print(f"  Dimensión latente  : {emb.shape[1]}")
print(f"  Modelo             : sentence-transformers/all-MiniLM-L6-v2")
print(f"  Similitud          : cosine (inner product sobre vectores normalizados)")

RESUMEN DEL PIPELINE DE EMBEDDINGS
  ✓ df_final_embeddings.parquet            146.5 MB  — DataFrame limpio con embedding_text e imágenes
  ✓ recipe_embeddings.npy                   88.8 MB  — Embeddings numpy (n × 384, float32, normalizado)
  ✓ recipe_faiss.index                      88.8 MB  — Índice FAISS IndexFlatIP para búsqueda exacta

  Recetas procesadas : 60,614
  Dimensión latente  : 384
  Modelo             : sentence-transformers/all-MiniLM-L6-v2
  Similitud          : cosine (inner product sobre vectores normalizados)
